In [1]:
from multiprocessing import Process, Manager
from threading import Thread
import datetime
import time
import pandas as pd
import websocket
import json
import traceback

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [2]:
data = {"op": "subscribe", "args": []}
both_listed_okx_symbols = ["BTC-USDT-SWAP", "ETH-USDT-SWAP", "XRP-USDT-SWAP", "EOS-USDT-SWAP"]
for symbol in both_listed_okx_symbols:
    data["args"].append({"channel": "funding-rate", "instId": symbol})
data

{'op': 'subscribe',
 'args': [{'channel': 'funding-rate', 'instId': 'BTC-USDT-SWAP'},
  {'channel': 'funding-rate', 'instId': 'ETH-USDT-SWAP'},
  {'channel': 'funding-rate', 'instId': 'XRP-USDT-SWAP'},
  {'channel': 'funding-rate', 'instId': 'EOS-USDT-SWAP'}]}

In [3]:
def okx_websocket(okx_result_dict, data):
    last_timestamp = [time.time()]
    def on_message(ws, message):
        last_timestamp[0] = time.time()
        if message == 'pong':
            # print(f"pong received from the server!") # TEST
            pass
        else:
            try:
                message_dict = json.loads(message)
                if 'data' in message_dict.keys():
                    print(f"timestamp: {last_timestamp[0]}, {message_dict}") # TEST
                    okx_result_dict[message_dict['data'][0]['instId']] = {**message_dict['data'][0], "last_updated_time": datetime.datetime.now()}
            except Exception as e:
                # price_websocket_logger.info(f"Error occured while json.loads message: {traceback.format_exc()}, message: {message}")
                print(f"Error occured while json.loads message: {traceback.format_exc()}, message: {message}") # TEST

    def on_error(ws, error):
        # price_websocket_logger.error(f'okx_ticker_websocket|okx_websocket on_error executed!\n Error: {error}')
        print(f'okx_ticker_websocket|okx_websocket on_error executed!\n Error: {error}') # TEST
        pass

    def on_close(ws, close_status_code, close_msg):
        # price_websocket_logger.info(f"okx_ticker_websocket|\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}")
        print(f"okx_ticker_websocket|\n\n### closed ###\nclose_msg: {close_msg}\nclose_status_code: {close_status_code}") # TEST

    def on_open(ws):
        # print(f'okx_ticker_websocket on_open started')
        # price_websocket_logger.info('okx_ticker_websocket|okx_websocket started')
        print('okx_ticker_websocket|okx_websocket started') # TEST
        ws.send(json.dumps(data))

    def ping(ws):
        while True:
            time.sleep(0.5)
            if time.time() - last_timestamp[0] > 25:
                try:
                    ws.send('ping')
                    # print(f"ping sent to the server!") # TEST
                except Exception as e:
                    # price_websocket_logger.error(f"Error while trying to ping: {traceback.format_exc()}")
                    print(f"Error while trying to ping: {traceback.format_exc()}") # TEST

    # websocket.enableTrace(False)
    ws = websocket.WebSocketApp("wss://ws.okx.com:8443/ws/v5/public",
                            on_open=on_open,
                            on_message=on_message,
                            on_error=on_error,
                            on_close=on_close)
    ping_thread = Thread(target=ping, args=(ws,))
    ping_thread.start()
    ws.run_forever()

In [4]:
funding_dict = {}
websocket_thread = Thread(target=okx_websocket, args=(funding_dict, data))
websocket_thread.start()

okx_ticker_websocket|okx_websocket started
timestamp: 1687501467.9802048, {'arg': {'channel': 'funding-rate', 'instId': 'BTC-USDT-SWAP'}, 'data': [{'fundingRate': '0.0000077462654925', 'fundingTime': '1687507200000', 'instId': 'BTC-USDT-SWAP', 'instType': 'SWAP', 'nextFundingRate': '0.0001500215257157', 'nextFundingTime': '1687536000000'}]}
timestamp: 1687501467.9803677, {'arg': {'channel': 'funding-rate', 'instId': 'ETH-USDT-SWAP'}, 'data': [{'fundingRate': '0.0000706151175557', 'fundingTime': '1687507200000', 'instId': 'ETH-USDT-SWAP', 'instType': 'SWAP', 'nextFundingRate': '-0.0000107843493057', 'nextFundingTime': '1687536000000'}]}
timestamp: 1687501467.980489, {'arg': {'channel': 'funding-rate', 'instId': 'EOS-USDT-SWAP'}, 'data': [{'fundingRate': '0.0000857690378264', 'fundingTime': '1687507200000', 'instId': 'EOS-USDT-SWAP', 'instType': 'SWAP', 'nextFundingRate': '0.0000859014657150', 'nextFundingTime': '1687536000000'}]}
timestamp: 1687501467.9806032, {'arg': {'channel': 'fundi

In [33]:
funding_df = pd.DataFrame(funding_dict).T.reset_index(drop=True)
funding_df.loc[:, ['fundingRate', 'fundingTime', 'nextFundingRate', 'nextFundingTime']] = funding_df.loc[:, ['fundingRate', 'fundingTime', 'nextFundingRate', 'nextFundingTime']].astype(float)
funding_df.loc[:, 'fundingTime'] = funding_df['fundingTime'].apply(lambda x: datetime.datetime.fromtimestamp(x/1000))
funding_df.loc[:, 'nextFundingTime'] = funding_df['nextFundingTime'].apply(lambda x: datetime.datetime.fromtimestamp(x/1000))
funding_df.loc[:, 'last_updated_time'] = funding_df['last_updated_time'].dt.to_pydatetime()
funding_df = funding_df.rename(columns={'fundingTime':'fundingtime', 'nextFundingTime':'nextfundingtime', 'nextFundingRate':"nextfundingrate", 'instId':'okx_symbol'})
funding_df['exchange'] = 'okx'
funding_df['rec'] = funding_df['okx_symbol'] + funding_df['fundingtime'].astype(str) + funding_df['exchange']
funding_df

,fundingRate,fundingtime,okx_symbol,instType,nextfundingrate,nextfundingtime,last_updated_time,exchange,rec
0,0.000008,2023-06-23 17:00:00,BTC-USDT-SWAP,SWAP,0.00015,2023-06-24 01:00:00,2023-06-23 15:30:16.387456,okx,BTC-USDT-SWAP2023-06-23 17:00:00okx
1,0.000071,2023-06-23 17:00:00,ETH-USDT-SWAP,SWAP,-0.000013,2023-06-24 01:00:00,2023-06-23 15:30:16.411379,okx,ETH-USDT-SWAP2023-06-23 17:00:00okx
2,0.000086,2023-06-23 17:00:00,EOS-USDT-SWAP,SWAP,0.000083,2023-06-24 01:00:00,2023-06-23 15:30:16.341998,okx,EOS-USDT-SWAP2023-06-23 17:00:00okx
3,0.000063,2023-06-23 17:00:00,XRP-USDT-SWAP,SWAP,0.000109,2023-06-24 01:00:00,2023-06-23 15:30:16.531608,okx,XRP-USDT-SWAP2023-06-23 17:00:00okx


In [34]:
funding_df['last_updated_time']#.iloc[0]

0   2023-06-23 15:30:16.387456
1   2023-06-23 15:30:16.411379
2   2023-06-23 15:30:16.341998
3   2023-06-23 15:30:16.531608
Name: last_updated_time, dtype: datetime64[ns]

In [35]:
funding_df['fundingtime']#.iloc[0]

0    2023-06-23 17:00:00
1    2023-06-23 17:00:00
2    2023-06-23 17:00:00
3    2023-06-23 17:00:00
Name: fundingtime, dtype: object

In [50]:
# print(f"old_fund_df['rec']: {old_fund_df['rec']}")
for row_tup in funding_df.iterrows():
    row = row_tup[1]
    # print(f"row['rec']: {row['rec']}")
    if row['rec'] not in old_fund_df['rec'].to_list():
        # Insert the new funding rate
        print(f"new funding rate detected: {row['okx_symbol']}, {row['fundingtime']}, {row['fundingRate']}")
    else:
        # Update the old_fund_df
        print(f"old funding rate detected: {row['okx_symbol']}, {row['fundingtime']}, {row['fundingRate']}")


old funding rate detected: ETH-USDT-SWAP, 2023-06-22 17:00:00, 0.0001680294781585
old funding rate detected: BTC-USDT-SWAP, 2023-06-22 17:00:00, 0.0003011904160944
old funding rate detected: EOS-USDT-SWAP, 2023-06-22 17:00:00, 2.93040464492e-05
old funding rate detected: XRP-USDT-SWAP, 2023-06-22 17:00:00, 6.5020623428e-05


timestamp: 1687408383.278629, {'arg': {'channel': 'funding-rate', 'instId': 'EOS-USDT-SWAP'}, 'data': [{'fundingRate': '0.0000293040464492', 'fundingTime': '1687420800000', 'instId': 'EOS-USDT-SWAP', 'instType': 'SWAP', 'nextFundingRate': '0.0000895136854518', 'nextFundingTime': '1687449600000'}]}
timestamp: 1687408383.3272643, {'arg': {'channel': 'funding-rate', 'instId': 'BTC-USDT-SWAP'}, 'data': [{'fundingRate': '0.0003011904160944', 'fundingTime': '1687420800000', 'instId': 'BTC-USDT-SWAP', 'instType': 'SWAP', 'nextFundingRate': '0.0000534060242362', 'nextFundingTime': '1687449600000'}]}
timestamp: 1687408383.344015, {'arg': {'channel': 'funding-rate', 'instId': 'ETH-USDT-SWAP'}, 'data': [{'fundingRate': '0.0001680294781585', 'fundingTime': '1687420800000', 'instId': 'ETH-USDT-SWAP', 'instType': 'SWAP', 'nextFundingRate': '0.0000404931536320', 'nextFundingTime': '1687449600000'}]}
timestamp: 1687408383.4452174, {'arg': {'channel': 'funding-rate', 'instId': 'XRP-USDT-SWAP'}, 'data':

[]